In [8]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def preprocess(train_file, test_file):
    train = np.load(file=train_file, allow_pickle=True)
    test = np.load(file=test_file, allow_pickle=True)

    X_train, y_train = train['x'], train['y']
    print(X_train.shape, X_train.dtype, y_train.shape, y_train.dtype)
    X_test, y_test = test['x'], test['y']
    print(X_test.shape, X_test.dtype, y_test.shape, y_test.dtype)

    print(len(np.unique(y_train)))

    transform = transforms.Compose([
        transforms.Resize((32, 32), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
    ])

    images_train = []
    for img in X_train:
        img = Image.fromarray(img)
        img = transform(img)
        images_train.append(img)


    images_test = []
    for img in X_test:
        img = Image.fromarray(img)
        img = transform(img)
        images_test.append(img)

    X_train = torch.stack(images_train)
    X_test  = torch.stack(images_test)

    print(X_train[0].shape)

    return (X_train, y_train), (X_test, y_test)

(X_train, y_train), (X_test, y_test) = preprocess("Numarul6-train/2_6_train.npz", "Numarul6-test/2_6_test.npz")

(28000,) object (28000,) int64
(7000,) object (7000,) int64
7
torch.Size([3, 32, 32])


In [9]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.act1  = nn.ReLU()
        self.act2 = nn.LeakyReLU()

        # IN -> (3, 32, 32)
        self.con1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1) # -> (32, 32, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2) # -> (32, 16, 16)
        self.con2 = nn.Conv2d(in_channels=32, out_channels=96, kernel_size=3, padding=1) # -> (96, 16, 16)
        self.pool2 = nn.MaxPool2d(kernel_size=2) # -> (96, 8, 8)
        self.con3 = nn.Conv2d(in_channels=96, out_channels=16, kernel_size=3, padding=1) # -> (16, 8, 8)
        self.pool3 = nn.MaxPool2d(kernel_size=2) # -> (16, 4, 4)
        self.con4 = nn.Conv2d(in_channels=16, out_channels=8, kernel_size=3, padding=1) # -> (8, 4, 4)
        self.pool4 = nn.MaxPool2d(kernel_size=2) # -> (8, 2, 2)

        self.flat = nn.Flatten()

        self.fc1 = nn.Linear(in_features=8 * 2 * 2, out_features=128)
        self.drop1 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(in_features=128, out_features=64)
        self.drop2 = nn.Dropout(p=0.2)
        self.fc3 = nn.Linear(in_features=64, out_features=7)



    def forward(self, x):
        x = self.pool1(self.act1(self.con1(x)))
        x = self.pool2(self.act2(self.con2(x)))
        x = self.pool3(self.act2(self.con3(x)))
        x = self.pool4(self.act1(self.con4(x)))
        
        x = self.flat(x)

        x = self.drop1(self.act2(self.fc1(x)))
        x = self.drop2(self.act2(self.fc2(x)))
        x = self.fc3(x)

        return x

model = Model()

In [10]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def test(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}"
    )

torch.save(model.state_dict(), "model2.pth")

/tmp/ipykernel_48402/1683137203.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_train, dtype=torch.float32),
/tmp/ipykernel_48402/1683137203.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_test, dtype=torch.float32),


Epoch 1/15 | Train Loss: 1.8885 | Train Acc: 19.04% | Test Loss: 1.7162 | Test Acc: 29.53%
Epoch 2/15 | Train Loss: 1.2366 | Train Acc: 53.72% | Test Loss: 0.9131 | Test Acc: 68.59%
Epoch 3/15 | Train Loss: 0.7922 | Train Acc: 72.95% | Test Loss: 0.7025 | Test Acc: 76.17%
Epoch 4/15 | Train Loss: 0.6552 | Train Acc: 77.92% | Test Loss: 0.5857 | Test Acc: 81.21%
Epoch 5/15 | Train Loss: 0.5708 | Train Acc: 80.96% | Test Loss: 0.5817 | Test Acc: 81.27%
Epoch 6/15 | Train Loss: 0.5235 | Train Acc: 82.85% | Test Loss: 0.5692 | Test Acc: 81.43%
Epoch 7/15 | Train Loss: 0.4881 | Train Acc: 83.94% | Test Loss: 0.5274 | Test Acc: 82.89%
Epoch 8/15 | Train Loss: 0.4579 | Train Acc: 85.16% | Test Loss: 0.5041 | Test Acc: 84.06%
Epoch 9/15 | Train Loss: 0.4409 | Train Acc: 85.64% | Test Loss: 0.4812 | Test Acc: 84.51%
Epoch 10/15 | Train Loss: 0.4151 | Train Acc: 86.68% | Test Loss: 0.4806 | Test Acc: 84.79%
Epoch 11/15 | Train Loss: 0.4054 | Train Acc: 86.76% | Test Loss: 0.4701 | Test Acc: 85.1